# K-Prototypes Deep-Dive (CircuitNet-N28)

Companion to `partitioning_analysis.ipynb`. Same encoding + weight schema, but
focused on validating the P3 partitioner itself rather than comparing it against
P0/P1/P2:

1. **k sweep** — silhouette (subsampled), inertia (elbow), Davies-Bouldin over
   `k ∈ [2, 14]`. Pick k where at least two of the three curves agree.
2. **Design-purity check** — cluster × `design_name` contingency table +
   bias-corrected Cramér's V. If V(cluster, design_name) > 0.9 the clustering
   is a design-based split in disguise. Recovery: drop `design_name` weight
   from 2.0 → 1.0 (or lower) and re-check; side-by-side report below.
3. **Feature importance** — η² for interval / ordinal features, Cramér's V for
   nominal features, both against the cluster label. Ranks which knobs actually
   drive the boundary.
4. **Cluster visualization** — 2D scatter on the top-2 informative axes plus a
   PCA projection of the full encoded space, both colored by cluster and by
   `design_name`.

# Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
from scipy import stats as scipy_stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 11,
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.8,
    'axes.facecolor': '#f5f5f5',
    'figure.facecolor': 'white',
})

sys.path.insert(0, os.path.abspath('.'))
from partitioning import KPrototypesPartitioner

FEATURE_DIR = '../routability_ir_drop_prediction/training_set_N28/DRC/feature'
SEED = 42
SILHOUETTE_SAMPLE_SIZE = 2000  # keep silhouette memory ~O(sample^2)
print('Imports OK.')

# Load metadata + derive features

Identical to `partitioning_analysis.ipynb`. Same parser, same derived columns
(all keyed directly on `design_name`), so the encoded matrix here is
byte-for-byte the one P3 sees.

In [ ]:
def parse_sample_name(filename: str) -> dict:
    basename = filename.replace('.npy', '')
    parts = basename.split('-')
    if parts[0].isdigit():
        parts = parts[1:]
    if len(parts) < 7:
        raise ValueError(f'Cannot parse: {filename}')
    for expected, token in zip(['c','u','m','p','f'], parts[-5:]):
        if not token.startswith(expected):
            raise ValueError(f'Bad token {token} in {filename}')
    c, u, m, p, f = parts[-5:]
    return {
        'design_name':      '-'.join(parts[:-6]),
        'macro_count':      parts[-6],
        'clock_ns':         float(c[1:]),
        'utilization':      float(u[1:]),
        'macro_placement':  m[1:],
        'power_mesh':       p[1:],
        'filler_insertion': f[1:],
        'filename':         filename,
    }

files = sorted(f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy'))
df_meta = pd.DataFrame([parse_sample_name(f) for f in files])
df_meta.loc[df_meta['macro_placement'].isin(['1','2','3','4']) == False, 'macro_placement'] = '1'

SIZE_MAP = {
    'zero-riscy-a': 'small', 'zero-riscy-b': 'small',
    'RISCY-a':      'small', 'RISCY-b':      'small',
    'RISCY-FPU-a':  'small', 'RISCY-FPU-b':  'small',
    'OpenC910-1':   'medium', 'Vortex-small': 'medium',
    'Vortex-large': 'large',  'NVDLA-small':  'large', 'NVDLA-large': 'large',
}
FILLER_MAP = {'0': 'after_routing', '1': 'after_placement'}

df_meta['size_class']    = df_meta['design_name'].map(SIZE_MAP).fillna('small')
df_meta['log_frequency'] = np.log10(1000.0 / df_meta['clock_ns'].astype(float))
df_meta['filler']        = df_meta['filler_insertion'].map(FILLER_MAP).fillna(df_meta['filler_insertion'])
df_meta['aspect_ratio']  = 1.0

print(f'Samples: {len(df_meta)}   Designs: {df_meta["design_name"].nunique()}')

# Encode once with the P3 weight schema

`KPrototypesPartitioner.encode(df)` returns the weighted mixed-type matrix
without running k-means, so the k-sweep below reuses it for every k.

In [ ]:
FEATURE_SPECS = [
    ('design_name',      'nominal',  2.00),
    ('size_class',       'ordinal',  1.00),
    ('log_frequency',    'interval', 1.50),
    ('utilization',      'interval', 1.50),
    ('aspect_ratio',     'interval', 1.00),
    ('power_mesh',       'nominal',  0.75),
    ('macro_placement',  'nominal',  0.75),
    ('filler',           'nominal',  0.50),
]
ORDINAL_ORDERS = {'size_class': ['small', 'medium', 'large']}

# n_partitions=2 is a placeholder; encode() doesn't cluster.
encoder = KPrototypesPartitioner(
    n_partitions=2,
    feature_specs=FEATURE_SPECS,
    ordinal_orders=ORDINAL_ORDERS,
    seed=SEED,
)
X = encoder.encode(df_meta)
print(f'Encoded matrix shape: {X.shape}')
print(f'Dropped features (missing / constant on N28): {encoder.dropped_features_}')

dropped_names = {name for name, _ in encoder.dropped_features_}
RETAINED_SPECS = [s for s in FEATURE_SPECS if s[0] not in dropped_names]
print('Retained features (col, kind, weight):')
for col, kind, w in RETAINED_SPECS:
    print(f'  {col:18s} {kind:9s}  w={w}')

# 1. k sweep

Three complementary metrics on the same encoded matrix:

- **Silhouette** (higher = better; sample of 2000 rows so we don't build a
  10k×10k distance matrix). Peaks at the "most-separated" k.
- **Inertia** (lower = better). Look for the elbow.
- **Davies-Bouldin** (lower = better). Complements silhouette; when they
  agree, the choice is stable.

N28 has 6 designs and 3 clock values, so the natural interpretable k values
are ≈ 3, 6, 9, 12. Sweeping [2, 14] covers all of them plus a couple beyond
12 to confirm the metric doesn't keep improving.

In [ ]:
K_VALUES = list(range(2, 15))

records = []
labels_by_k = {}
for k in K_VALUES:
    km = KMeans(n_clusters=k, n_init=20, max_iter=300, random_state=SEED)
    lbl = km.fit_predict(X)
    labels_by_k[k] = lbl
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X), size=min(SILHOUETTE_SAMPLE_SIZE, len(X)), replace=False)
    sil = float(silhouette_score(X[idx], lbl[idx]))
    dbi = float(davies_bouldin_score(X, lbl))
    records.append({'k': k, 'silhouette': sil, 'inertia': float(km.inertia_), 'DBI': dbi})

sweep = pd.DataFrame(records).set_index('k')
print(sweep.round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, metric, ylabel, better in [
    (axes[0], 'silhouette', 'Silhouette (subsample=2000)',  'higher = better'),
    (axes[1], 'inertia',    'Inertia (SSE)',                'lower = better (elbow)'),
    (axes[2], 'DBI',        'Davies-Bouldin index',         'lower = better'),
]:
    ax.plot(sweep.index, sweep[metric], marker='o', color='#2c3e50', linewidth=1.8)
    ax.set_xlabel('k')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{metric}  ({better})')
    ax.set_xticks(sweep.index)
    if metric == 'silhouette':
        best = sweep[metric].idxmax()
    else:
        best = sweep[metric].idxmin()
    ax.axvline(best, linestyle='--', color='#c0392b', alpha=0.6,
               label=f'best = {best}')
    ax.legend()
plt.tight_layout()
plt.show()

vote = {
    'silhouette (max)':      int(sweep['silhouette'].idxmax()),
    'inertia elbow (max Δ²)': int((sweep['inertia'].diff().diff().abs()).idxmax()),
    'DBI (min)':             int(sweep['DBI'].idxmin()),
}
print('\nCandidate k per criterion:')
for name, k in vote.items():
    print(f'  {name:25s} -> k = {k}')

# 2. Fit at the chosen k and inspect cluster sizes

We keep `k = 12` for parity with the P0/P1/P2 comparison in
`partitioning_analysis.ipynb` (P1 is 6 designs × 2 clock bins). If the sweep
above prefers a different k, adjust `K_CHOSEN` and re-run the remaining cells.

In [ ]:
K_CHOSEN = 12
labels = labels_by_k[K_CHOSEN]
df_meta['cluster'] = labels

sizes = pd.Series(labels).value_counts().sort_index()
print(f'Cluster sizes at k={K_CHOSEN}:')
print(sizes.to_string())
print(f'  imbalance max/min = {sizes.max()/max(sizes.min(),1):.2f}x')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(sizes.index, sizes.values, color=mcm.get_cmap('tab20')(np.arange(len(sizes)) % 20),
       alpha=0.9, edgecolor='white')
for i, v in enumerate(sizes.values):
    ax.text(sizes.index[i], v + 20, str(v), ha='center', fontsize=9)
ax.set_xlabel('cluster id')
ax.set_ylabel('sample count')
ax.set_title(f'Cluster sizes  (k={K_CHOSEN})')
plt.tight_layout()
plt.show()

# 3. Design-purity check (Cramér's V)

The primary concern with `design_name` weighted at 2.0 is that clusters may
end up as design-based splits with extra steps.

Reported:

- Bias-corrected Cramér's V for (cluster × `design_name`). Rule of thumb:
  - **V ≥ 0.90** → clusters are effectively a design partition. Either accept
    and say so explicitly, or lower the `design_name` weight (Section 4).
  - **V ∈ [0.60, 0.90]** → design dominates the boundary but non-design knobs
    still shape it.
  - **V < 0.60** → design contributes but doesn't monopolize; the
    physical-design knobs are meaningfully in play.
- Row-normalized contingency table `P(design_name | cluster)` — the most useful
  qualitative view. A block-diagonal pattern (after cluster re-ordering) is
  the visual analogue of high V.

In [ ]:
def cramers_v(x, y, bias_correction=True):
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = scipy_stats.chi2_contingency(ct)
    n = ct.values.sum()
    r, k = ct.shape
    if min(r, k) < 2 or n == 0:
        return float('nan')
    if not bias_correction:
        return float(np.sqrt(chi2 / (n * (min(r, k) - 1))))
    phi2 = chi2 / n
    phi2_corr = max(0.0, phi2 - ((r - 1) * (k - 1)) / (n - 1))
    r_c = r - ((r - 1) ** 2) / (n - 1)
    k_c = k - ((k - 1) ** 2) / (n - 1)
    denom = min(k_c - 1, r_c - 1)
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2_corr / denom))


def dominant_level_share(df, cluster_col, target_col):
    rows = []
    for c, group in df.groupby(cluster_col):
        vc = group[target_col].value_counts(normalize=True)
        rows.append({
            'cluster':          c,
            'size':             len(group),
            'dominant':         vc.index[0],
            'dominant_share':   round(float(vc.iloc[0]), 3),
        })
    return pd.DataFrame(rows).set_index('cluster')


v_design = cramers_v(df_meta['cluster'], df_meta['design_name'])
print(f"Cramer's V (cluster x design_name) = {v_design:.3f}")

def _interp(v):
    if v >= 0.90: return 'DEGENERATE (design-based split with extra steps)'
    if v >= 0.60: return 'design-dominated but non-design knobs still shape boundaries'
    return 'design contributes but is not monopolizing'
print('Interpretation:', _interp(v_design))

print('\nDominant design per cluster:')
print(dominant_level_share(df_meta, 'cluster', 'design_name').to_string())

# Row-normalized contingency P(design_name|cluster) heatmap
fig, ax = plt.subplots(figsize=(12, 6))
ct = pd.crosstab(df_meta['cluster'], df_meta['design_name'])
row_norm = ct.div(ct.sum(axis=1), axis=0)
im = ax.imshow(row_norm.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_yticks(range(len(row_norm.index)))
ax.set_yticklabels([f'c{c}' for c in row_norm.index])
ax.set_xticks(range(len(row_norm.columns)))
ax.set_xticklabels(row_norm.columns, rotation=35, ha='right')
for i in range(row_norm.shape[0]):
    for j in range(row_norm.shape[1]):
        v = row_norm.values[i, j]
        if v >= 0.05:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v > 0.55 else 'black', fontsize=8)
ax.set_title(f'P(design_name | cluster)   V = {v_design:.3f}')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

# 4. Weight sensitivity: lower `design_name` from 2.0 → 1.0 → 0.5

If Section 3 flagged degeneracy (V > 0.90 on design_name), this section shows
what happens when we push the design weight down. The other weights are kept
fixed at the user-supplied values; we sweep only the `design_name` weight.

Report per weight setting:

- Cramér's V (cluster × design_name) → drops as the design weight drops
- Cramér's V (cluster × log_frequency binned) → typically rises: the persona
  knob gets more room to influence the boundary
- Silhouette on the new encoded matrix — the clustering may become less crisp
  when the strongest signal is de-emphasized (this is the price of forcing
  physical-knob influence)

In [ ]:
DESIGN_WEIGHTS = [2.00, 1.00, 0.50]

def specs_with_design_weight(w):
    return [(c, kind, wt if c != 'design_name' else w) for c, kind, wt in FEATURE_SPECS]

sweep_rows = []
labels_by_w = {}
for w in DESIGN_WEIGHTS:
    enc = KPrototypesPartitioner(
        n_partitions=2,
        feature_specs=specs_with_design_weight(w),
        ordinal_orders=ORDINAL_ORDERS,
        seed=SEED,
    )
    X_w = enc.encode(df_meta)
    km = KMeans(n_clusters=K_CHOSEN, n_init=20, max_iter=300, random_state=SEED)
    lbl = km.fit_predict(X_w)
    labels_by_w[w] = lbl
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X_w), size=min(SILHOUETTE_SAMPLE_SIZE, len(X_w)), replace=False)
    sil = float(silhouette_score(X_w[idx], lbl[idx]))
    v_des = cramers_v(pd.Series(lbl), df_meta['design_name'])
    # Binned log_frequency and utilization: 3 quantile bins so V is comparable across weights.
    freq_bins = pd.qcut(df_meta['log_frequency'], q=3, duplicates='drop').astype(str)
    util_bins = pd.qcut(df_meta['utilization'],   q=3, duplicates='drop').astype(str)
    v_freq = cramers_v(pd.Series(lbl), freq_bins)
    v_util = cramers_v(pd.Series(lbl), util_bins)
    sweep_rows.append({
        'design_weight':           w,
        "V(cluster,design)":       round(v_des, 3),
        "V(cluster,log_freq_bin)": round(v_freq, 3),
        "V(cluster,util_bin)":     round(v_util, 3),
        'silhouette':              round(sil, 3),
        'inertia':                 round(float(km.inertia_), 2),
    })

sweep_df = pd.DataFrame(sweep_rows).set_index('design_weight')
print(sweep_df.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
for col, color in [
    ("V(cluster,design)",       '#c0392b'),
    ("V(cluster,log_freq_bin)", '#2980b9'),
    ("V(cluster,util_bin)",     '#16a085'),
]:
    ax.plot(sweep_df.index, sweep_df[col], marker='o', label=col, color=color)
ax.set_xlabel('design_name weight')
ax.set_ylabel("Cramer's V (cluster x ...)")
ax.set_title('Where cluster boundaries land as the design_name weight decreases')
ax.set_xticks(DESIGN_WEIGHTS)
ax.axhline(0.90, color='#c0392b', linestyle='--', alpha=0.5, label='degenerate threshold')
ax.axhline(0.60, color='#7f8c8d', linestyle='--', alpha=0.5, label='dominance threshold')
ax.legend(fontsize=8, loc='center right')
plt.tight_layout()
plt.show()

# 5. Feature importance for the chosen clustering

Which knobs actually drive the boundary at the primary configuration
(`k=K_CHOSEN`, original weights)?

- Numeric features (`log_frequency`, `utilization`) → η² (fraction of variance
  in the feature explained by the cluster label).
- Categorical / ordinal features (`design_name`, `size_class`, `power_mesh`,
  `macro_placement`, `filler`) → bias-corrected Cramér's V vs cluster.

Both metrics live on [0, 1] so they can be ranked side-by-side, with the
important caveat that they aren't directly comparable in magnitude across
types — use them to spot the top-2/top-3 axes rather than to make fine
distinctions.

In [ ]:
def eta_squared(cat_series, num_series):
    d = pd.DataFrame({'cat': cat_series.values,
                      'num': pd.to_numeric(num_series, errors='coerce').values}).dropna()
    grand = d['num'].mean()
    ss_tot = ((d['num'] - grand) ** 2).sum()
    if ss_tot == 0:
        return float('nan')
    ss_bet = sum(len(g) * (g['num'].mean() - grand) ** 2 for _, g in d.groupby('cat'))
    return float(ss_bet / ss_tot)

labels = labels_by_k[K_CHOSEN]
clu = pd.Series(labels, name='cluster')

rows = []
for col, kind, w in RETAINED_SPECS:
    if kind == 'interval':
        assoc = eta_squared(clu, df_meta[col])
        metric = 'eta^2'
    else:
        assoc = cramers_v(clu, df_meta[col])
        metric = 'Cramer V'
    rows.append({'feature': col, 'kind': kind, 'weight': w,
                 'metric': metric, 'assoc': round(assoc, 3)})
imp = pd.DataFrame(rows).sort_values('assoc', ascending=False).reset_index(drop=True)
print(imp.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ['#c0392b' if k == 'nominal' else
          '#16a085' if k == 'interval' else
          '#8e44ad' for k in imp['kind']]
ax.barh(imp['feature'], imp['assoc'], color=colors, alpha=0.9, edgecolor='white')
ax.invert_yaxis()
for i, v in enumerate(imp['assoc']):
    ax.text(v + 0.005, i, f'{v:.2f}', va='center', fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel("association with cluster (eta^2 for interval, Cramer's V for nominal/ordinal)")
ax.set_title(f'Feature importance at k={K_CHOSEN}')
plt.tight_layout()
plt.show()

TOP_FEATURES = imp['feature'].tolist()[:3]
print(f'\nTop-3 features (used for visualization): {TOP_FEATURES}')

# 6. Cluster visualization

Three views:

1. **Top-2 informative-feature scatter** with jitter (the two design knobs at
   the top of Section 5). Colored by cluster; marker shape by `design_name`.
   Small jitter on discrete axes is added so overplotted points remain readable.
2. **Three pairwise scatter plots** across the top-3 features (2D projections
   of the informative subspace).
3. **PCA(2)** of the full weighted-encoded space, colored by cluster and,
   separately, by `design_name`. Visualizes how much of the design axis the
   two principal directions capture.

In [ ]:
def _to_numeric(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float).to_numpy(), None
    levels = sorted(series.astype(str).unique().tolist())
    code_map = {lvl: i for i, lvl in enumerate(levels)}
    return series.astype(str).map(code_map).to_numpy(dtype=float), levels

def _axis_values(df, col):
    return _to_numeric(df[col])

def _jitter(values, scale=0.08, seed=SEED):
    rng = np.random.default_rng(seed)
    span = np.ptp(values) if np.ptp(values) > 0 else 1.0
    return values + rng.uniform(-scale, scale, size=len(values)) * span * 0.05

def scatter_cluster(ax, df, xcol, ycol, labels, group_col='design_name'):
    xv, xlvls = _axis_values(df, xcol)
    yv, ylvls = _axis_values(df, ycol)
    xv, yv = _jitter(xv, seed=SEED), _jitter(yv, seed=SEED + 1)
    cmap = mcm.get_cmap('tab20', max(int(labels.max()) + 1, 3))
    groups = sorted(df[group_col].astype(str).unique().tolist())
    markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', 'h', '<', '>', 'p']
    for gi, g in enumerate(groups):
        m = (df[group_col].astype(str) == g).to_numpy()
        if not m.any():
            continue
        ax.scatter(xv[m], yv[m], c=labels[m], cmap=cmap,
                   marker=markers[gi % len(markers)], s=14, alpha=0.6,
                   edgecolor='none', label=g,
                   vmin=0, vmax=max(int(labels.max()), 1))
    ax.set_xlabel(xcol); ax.set_ylabel(ycol)
    if xlvls is not None:
        ax.set_xticks(range(len(xlvls))); ax.set_xticklabels(xlvls, rotation=25, ha='right', fontsize=8)
    if ylvls is not None:
        ax.set_yticks(range(len(ylvls))); ax.set_yticklabels(ylvls, fontsize=8)

labels = labels_by_k[K_CHOSEN]

# View 1: top-2 pairwise scatter
top2 = TOP_FEATURES[:2]
fig, ax = plt.subplots(figsize=(10, 6))
scatter_cluster(ax, df_meta, top2[0], top2[1], labels)
ax.set_title(f'Cluster scatter (top-2 features)  color=cluster, marker=design_name')
ax.legend(fontsize=8, loc='center left', bbox_to_anchor=(1.0, 0.5), title='design_name')
plt.tight_layout()
plt.show()

# View 2: three pairwise scatters (top-3)
if len(TOP_FEATURES) >= 3:
    pairs = [(TOP_FEATURES[0], TOP_FEATURES[1]),
             (TOP_FEATURES[0], TOP_FEATURES[2]),
             (TOP_FEATURES[1], TOP_FEATURES[2])]
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    for ax, (xc, yc) in zip(axes, pairs):
        scatter_cluster(ax, df_meta, xc, yc, labels)
        ax.set_title(f'{xc}  vs  {yc}')
    axes[-1].legend(fontsize=8, loc='center left', bbox_to_anchor=(1.0, 0.5), title='design_name')
    plt.tight_layout()
    plt.show()

# View 3: PCA(2) on encoded space
pca = PCA(n_components=2, random_state=SEED)
Z = pca.fit_transform(X)
print(f'PCA explained variance: {pca.explained_variance_ratio_.round(3).tolist()} '
      f'(total {pca.explained_variance_ratio_.sum():.3f})')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cmap_c = mcm.get_cmap('tab20', max(int(labels.max()) + 1, 3))
sc1 = axes[0].scatter(Z[:, 0], Z[:, 1], c=labels, cmap=cmap_c, s=8, alpha=0.65,
                       edgecolor='none', vmin=0, vmax=max(int(labels.max()), 1))
axes[0].set_title(f'PCA(2) of encoded space  colored by cluster (k={K_CHOSEN})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(sc1, ax=axes[0], label='cluster id')

designs = sorted(df_meta['design_name'].astype(str).unique().tolist())
des_codes = df_meta['design_name'].astype(str).map({d: i for i, d in enumerate(designs)}).to_numpy()
cmap_d = mcm.get_cmap('tab10', max(len(designs), 3))
sc2 = axes[1].scatter(Z[:, 0], Z[:, 1], c=des_codes, cmap=cmap_d, s=8, alpha=0.65,
                       edgecolor='none', vmin=0, vmax=max(len(designs) - 1, 1))
axes[1].set_title('PCA(2) of encoded space  colored by design_name')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
cbar = plt.colorbar(sc2, ax=axes[1], ticks=range(len(designs)))
cbar.set_ticklabels(designs)
cbar.set_label('design_name')

plt.tight_layout()
plt.show()

# 7. Cluster profile table

For quick prose in the thesis: per cluster, the modal category of every
nominal/ordinal feature and the mean of every numeric feature.

In [ ]:
profile_rows = []
for c, group in df_meta.groupby('cluster'):
    row = {'cluster': int(c), 'size': len(group)}
    for col, kind, _ in RETAINED_SPECS:
        if kind == 'interval':
            row[col] = round(float(group[col].mean()), 3)
        else:
            vc = group[col].astype(str).value_counts(normalize=True)
            row[col] = f'{vc.index[0]} ({vc.iloc[0]:.0%})'
    profile_rows.append(row)
profile_df = pd.DataFrame(profile_rows).set_index('cluster')
print(profile_df.to_string())